In [1]:
from gurobipy import Model, GRB, quicksum
import pandas as pd
import numpy as np
import os

In [2]:
income_df = pd.read_csv("./ChildCareDeserts_Data/avg_individual_income.csv")
employment_df = pd.read_csv("./ChildCareDeserts_Data/employment_rate.csv")
population_df = pd.read_csv("./ChildCareDeserts_Data/population.csv")
childcare_df = pd.read_csv("./ChildCareDeserts_Data/child_care_regulated.csv")
location_df = pd.read_csv("./ChildCareDeserts_Data/potential_locations.csv")

In [3]:
# ---------- Helpers ----------
def normalize_zip(df):
    for col in df.columns:
        if "zip" in col.lower():
            df = df.rename(columns={col: "ZIP"})
            break
    df["ZIP"] = df["ZIP"].astype(str).str.zfill(5)
    return df


def first_col(df, keys):
    for c in df.columns:
        lc = c.lower()
        if any(k in lc for k in keys):
            return c
    return None

In [4]:
# ---------- normalize zips ----------
income_df = normalize_zip(income_df)
employment_df = normalize_zip(employment_df)
population_df = normalize_zip(population_df)
childcare_df = normalize_zip(childcare_df)
location_df = normalize_zip(location_df)

In [5]:
# ---------- current capacity per facility & per ZIP ----------
if "total_capacity" in childcare_df.columns:
    childcare_df["fac_capacity"] = childcare_df["total_capacity"]
else:
    cap_cols = [c for c in childcare_df.columns if "capacity" in c.lower()]
    if len(cap_cols) == 0:
        raise ValueError("No capacity columns found in child_care_regulated.csv")
    childcare_df["fac_capacity"] = childcare_df[cap_cols].sum(axis=1, numeric_only=True)

In [6]:
# Best-effort existing 0–5 capacity (if present)
u5_cols = [
    c
    for c in childcare_df.columns
    if any(k in c.lower() for k in ["0-5", "0–5", "infant", "toddler", "preschool"])
]
if len(u5_cols) > 0:
    childcare_df["fac_u5_capacity"] = childcare_df[u5_cols].sum(
        axis=1, numeric_only=True
    )
else:
    childcare_df["fac_u5_capacity"] = 0.0

In [7]:
# Per-ZIP existing totals
zip_capacity = (
    childcare_df.groupby("ZIP", as_index=False)["fac_capacity"]
    .sum()
    .rename(columns={"fac_capacity": "current_capacity"})
)
zip_u5_capacity = (
    childcare_df.groupby("ZIP", as_index=False)["fac_u5_capacity"]
    .sum()
    .rename(columns={"fac_u5_capacity": "current_u5_capacity"})
)

In [8]:
# ---------- population (0–5 and 0–12) ----------
# Try to find typical buckets: '0-5','5-9','10-14'
c_0_5 = [
    c
    for c in population_df.columns
    if "0-5" in c.lower() or "0–5" in c.lower() or "under 5" in c.lower()
]
c_5_9 = [c for c in population_df.columns if "5-9" in c.lower() or "5–9" in c.lower()]
c_10_14 = [
    c for c in population_df.columns if "10-14" in c.lower() or "10–14" in c.lower()
]


def safe_sum_cols(df, cols):
    return df[cols].sum(axis=1, numeric_only=True) if len(cols) > 0 else 0.0


population_df["pop_0_5"] = safe_sum_cols(population_df, c_0_5)
population_df["pop_5_9"] = safe_sum_cols(population_df, c_5_9)
population_df["pop_10_14"] = safe_sum_cols(population_df, c_10_14)

# 0–12 = (0–5) + (5–9) + (2/5)*(10–14)
population_df["pop_0_12"] = (
    population_df["pop_0_5"]
    + population_df["pop_5_9"]
    + 0.4 * population_df["pop_10_14"]
)

zip_population = population_df[["ZIP", "pop_0_5", "pop_0_12"]].copy()

In [9]:
# ---------- employment / income ----------
employment_df = employment_df.rename(columns={"employment rate": "employment_rate"})
income_df = income_df.rename(columns={"average income": "average_income"})

# ---------- per-ZIP base table ----------
zip_df = location_df[["ZIP"]].drop_duplicates()
zip_df = zip_df.merge(zip_capacity, on="ZIP", how="left")
zip_df = zip_df.merge(zip_u5_capacity, on="ZIP", how="left")
zip_df = zip_df.merge(zip_population, on="ZIP", how="left")
zip_df = zip_df.merge(employment_df[["ZIP", "employment_rate"]], on="ZIP", how="left")
zip_df = zip_df.merge(income_df[["ZIP", "average_income"]], on="ZIP", how="left")
zip_df[
    [
        "current_capacity",
        "current_u5_capacity",
        "pop_0_5",
        "pop_0_12",
        "employment_rate",
        "average_income",
    ]
] = zip_df[
    [
        "current_capacity",
        "current_u5_capacity",
        "pop_0_5",
        "pop_0_12",
        "employment_rate",
        "average_income",
    ]
].fillna(0)

ZIPs = zip_df["ZIP"].tolist()

In [10]:
# ---------- facility options (Problem 1) ----------
facility_options = [
    {"name": "S", "cap_total": 100, "cap_05_max": 50, "cost": 65000},
    {"name": "M", "cap_total": 200, "cap_05_max": 100, "cost": 95000},
    {"name": "L", "cap_total": 400, "cap_05_max": 200, "cost": 115000},
]

In [11]:
# ---------- MODEL ----------
m = Model("MinCostChildcare_P1")

# --- NEW FACILITIES: integer count per ZIP and size (Problem 1 allows any number anywhere) ---
n_build = {
    (z, opt["name"]): m.addVar(vtype=GRB.INTEGER, lb=0, name=f"build_{z}_{opt['name']}")
    for z in ZIPs
    for opt in facility_options
}

# Total new capacity and new under-5 capacity per ZIP
new_total = {
    z: quicksum(
        n_build[(z, opt["name"])] * opt["cap_total"] for opt in facility_options
    )
    for z in ZIPs
}
# decision variable for how many of the new slots are assigned to 0–5, capped by per-size max
new_u5 = {z: m.addVar(lb=0, vtype=GRB.CONTINUOUS, name=f"new_u5_{z}") for z in ZIPs}
for z in ZIPs:
    cap_u5_max = quicksum(
        n_build[(z, opt["name"])] * opt["cap_05_max"] for opt in facility_options
    )
    m.addConstr(new_u5[z] <= cap_u5_max, name=f"u5_new_cap_{z}")

Set parameter Username
Set parameter LicenseID to value 2722177
Academic license - for non-commercial use only - expires 2026-10-14


In [ ]:
# --- EXPANSION: per existing facility f (cap min(20%·n_f, 500)) ---
childcare_df = (
    childcare_df.reset_index(drop=True).reset_index().rename(columns={"index": "f_id"})
)
F = childcare_df["f_id"].tolist()
fac_zip = dict(zip(childcare_df["f_id"], childcare_df["ZIP"]))
n_f = dict(zip(childcare_df["f_id"], childcare_df["fac_capacity"]))

# ---------- EXPANSION  ----------
t1, t2, t3, x_f = {}, {}, {}, {}
b_base = {}

for f in F:
    cap120 = 1.2 * n_f[f]
    caplim = min(cap120, 500.0)  # max 120% ou +500 slots

    # segment limits: 0–50%, 50–100%, 100–120% (approximation linéaire)
    t1_ub = min(0.50 * n_f[f], caplim)
    t2_ub = min(max(0.0, 1.00 * n_f[f] - 0.50 * n_f[f]), max(0.0, caplim - t1_ub))
    t3_ub = max(0.0, caplim - t1_ub - t2_ub)

    t1[f] = m.addVar(lb=0, ub=t1_ub, vtype=GRB.CONTINUOUS, name=f"t1_{f}")
    t2[f] = m.addVar(lb=0, ub=t2_ub, vtype=GRB.CONTINUOUS, name=f"t2_{f}")
    t3[f] = m.addVar(lb=0, ub=t3_ub, vtype=GRB.CONTINUOUS, name=f"t3_{f}")
    x_f[f] = m.addVar(lb=0, ub=caplim, vtype=GRB.CONTINUOUS, name=f"x_{f}")
    b_base[f] = m.addVar(vtype=GRB.BINARY, name=f"base_{f}")

    m.addConstr(x_f[f] == t1[f] + t2[f] + t3[f], name=f"x_sum_{f}")

    m.addGenConstrIndicator(b_base[f], 1, x_f[f] >= n_f[f], name=f"ind_ge_{f}")
    m.addGenConstrIndicator(b_base[f], 0, x_f[f] <= n_f[f] - 1e-6, name=f"ind_lt_{f}")

exp_total = {z: quicksum(x_f[f] for f in F if fac_zip[f] == z) for z in ZIPs}

exp_u5 = {z: m.addVar(lb=0, vtype=GRB.CONTINUOUS, name=f"exp_u5_{z}") for z in ZIPs}
for z in ZIPs:
    m.addConstr(exp_u5[z] <= exp_total[z], name=f"exp_u5_le_exp_{z}")


In [13]:
# ---------- COVERAGE CONSTRAINTS ----------
EMPLOYMENT_T = 60
INCOME_T = 60000

zip_idx = {z: i for i, z in enumerate(ZIPs)}
zip_series = zip_df.set_index("ZIP")

for z in ZIPs:
    employ_rate = float(zip_series.loc[z, "employment_rate"])
    avg_income = float(zip_series.loc[z, "average_income"])
    pop_012 = float(zip_series.loc[z, "pop_0_12"])
    pop_05 = float(zip_series.loc[z, "pop_0_5"])
    curr_total = float(zip_series.loc[z, "current_capacity"])
    curr_u5 = float(zip_series.loc[z, "current_u5_capacity"])

    is_high = (employ_rate >= EMPLOYMENT_T) or (avg_income <= INCOME_T)
    required_total = 0.5 * pop_012 if is_high else (1.0 / 3.0) * pop_012

    # Total coverage (0–12)
    m.addConstr(
        curr_total + exp_total[z] + new_total[z] >= required_total,
        name=f"cov_total_{z}",
    )

    # 0–5 coverage (≥ 2/3 of 0–5 population)
    m.addConstr(
        curr_u5 + exp_u5[z] + new_u5[z] >= (2.0 / 3.0) * pop_05, name=f"cov_u5_{z}"
    )

In [14]:
# ---------- COSTS  ----------
# Ensure indicator meaning:
# b_base[f] = 1  ⇔  x_f >= n_f
for f in F:
    m.addGenConstrIndicator(b_base[f], True, x_f[f] >= n_f[f])
    m.addGenConstrIndicator(b_base[f], False, x_f[f] <= n_f[f] - 1e-6)


# Cost per slot BELOW 100% (depends on initial facility size)
def cost_per_slot_low(n):
    if n < 100:
        return 200
    elif n < 300:
        return 180
    else:
        return 160


expansion_cost_low = quicksum(
    cost_per_slot_low(n_f[f]) * x_f[f] * (1 - b_base[f])  # only applies when x_f < n_f
    for f in F
)

# Cost per slot ABOVE 100% (always 200 per slot)
expansion_cost_high = quicksum(
    200.0 * x_f[f] * b_base[f]  # only applies when x_f >= n_f
    for f in F
)

# Baseline fee triggered only if x_f >= n_f (fixed 20,000)
baseline_cost = quicksum(b_base[f] * 20000.0 for f in F)

# New builds (unchanged)
build_cost = quicksum(
    n_build[(z, opt["name"])] * opt["cost"] for z in ZIPs for opt in facility_options
)

# Cost for new 0–5 slots (unchanged)
equip_cost = quicksum(100.0 * (new_u5[z] + exp_u5[z]) for z in ZIPs)

# Final objective
m.setObjective(
    expansion_cost_low + expansion_cost_high + baseline_cost + build_cost + equip_cost,
    GRB.MINIMIZE,
)

In [15]:
# ---------- SOLVE ----------
m.optimize()

if m.status == GRB.OPTIMAL:
    print(f"\nOptimal total cost: ${m.objVal:,.2f}\n")

Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (mac64[arm] - Darwin 24.6.0 24G231)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 24220 rows, 88790 columns and 114278 nonzeros
Model fingerprint: 0xfd72345a
Model has 2051 quadratic objective terms
Model has 62416 simple general constraints
  62416 INDICATOR
Variable types: 66724 continuous, 22066 integer (15604 binary)
Coefficient statistics:
  Matrix range     [1e+00, 4e+02]
  Objective range  [1e+02, 1e+05]
  QObjective range [4e+01, 8e+01]
  Bounds range     [6e-01, 5e+02]
  RHS range        [3e-01, 6e+03]
  GenCon rhs range [1e-06, 9e+02]
  GenCon coe range [1e+00, 1e+00]
Presolve added 61930 rows and 0 columns
Presolve removed 0 rows and 4917 columns
Presolve time: 0.30s
Presolved: 88186 rows, 85909 columns, 202300 nonzeros
Presolved model has 824 SOS constraint(s)
Variable types: 17136 continuous, 68773 integer (66077 binary)
Found heuristic solution: objec